In [7]:

# ============================================================================
# CELL 1: IMPORTS AND ENVIRONMENT SETUP
# ============================================================================
# This cell loads all libraries and configures the runtime environment.
# The overall goal: solve the 1D viscous Burgers' equation using a
# Physics-Informed Neural Network (PINN), trained first with Adam (1st-order)
# and then with a Self-Scaled Broyden quasi-Newton method (2nd-order).
# ============================================================================

import os          # Filesystem operations (creating directories)
import math        # Standard math (sqrt, etc.) for weight initialization
from time import perf_counter  # High-resolution timer for benchmarking

import numpy as np              # NumPy: array math, random sampling
import scipy.io                 # SciPy I/O: load .mat files (reference solution)
from scipy.optimize import minimize   # SciPy's minimize — patched with self-scaled BFGS variants
from scipy.linalg import cholesky, LinAlgError  # Cholesky decomposition to verify positive-definiteness of Hessian

import torch                    # PyTorch: tensor operations, autograd, neural networks
import torch.nn as nn           # PyTorch neural network module (layers, loss functions)
from torch.nn.utils import parameters_to_vector, vector_to_parameters
# parameters_to_vector: flattens all network weights into a single 1-D vector (needed by SciPy)
# vector_to_parameters: writes a flat vector back into the network's parameter tensors

# Use float64 (double precision) everywhere — critical for second-order optimizers
# which need accurate gradients and Hessian approximations
torch.set_default_dtype(torch.float64)

# Fix random seed for reproducibility
torch.manual_seed(2)

# Use GPU if available, otherwise CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create a directory to save results (checkpoints, plots, etc.)
RESULTS_DIR = "results2"
os.makedirs(RESULTS_DIR, exist_ok=True)


In [10]:

# ============================================================================
# CELL 2: FULL PINN IMPLEMENTATION — MODEL, TRAINING, BFGS OPTIMIZATION
# ============================================================================
#
# THE PDE WE ARE SOLVING:
# -----------------------
# 1D Viscous Burgers' equation:
#
#     ∂u/∂t + u · ∂u/∂x = ν · ∂²u/∂x²
#
# where:
#   u(t,x) is the unknown velocity field
#   ν = 0.01/π is the viscosity (small ⇒ sharp gradients / shock formation)
#
# Domain:  x ∈ [-1, 1],  t ∈ [0, 1]
#
# Initial condition:   u(0, x) = -sin(πx)
# Boundary conditions: u(t, -1) = u(t, 1) = 0  (Dirichlet)
#
# PINN APPROACH:
# - A neural network N(t,x) → u approximates the solution.
# - The loss has 3 terms:
#   1. PDE residual loss:  mean[ (u_t + u·u_x - ν·u_xx)² ]  at interior collocation points
#   2. IC loss:            mean[ (N(0,x) - u₀(x))² ]         at initial-condition points
#   3. BC loss:            mean[ (N(t,-1) - 0)² + (N(t,1) - 0)² ]  at boundary points
# - The network is trained to make all three losses zero simultaneously.
# ============================================================================


# ========================
# SECTION 1: HYPERPARAMETERS
# ========================

scaling = 1          # Output scaling factor (multiplied to the network output; 1 = no scaling)
L1 = 20              # Number of neurons in each hidden layer

# Network architecture: input(2) → 20 → 20 → 20 → 20 → output(1)
# Input is (t, x), output is u(t,x)
layer_dims = (2, L1, L1, L1, L1, 1)

# Spatial domain: x ∈ [x0, xf] = [-1, 1], so L = xf - x0 = 2
L = 2        # Length of spatial domain
x0 = -1      # Left boundary
xf = x0 + L  # Right boundary = 1

tfinal = 1           # Final time
nu = 0.01 / np.pi   # Viscosity coefficient ν = 0.01/π ≈ 0.00318

# --- Training schedule ---
Nepochs_ADAM = 1000   # Number of Adam (first-order) warmup iterations
Nchange = 500         # Resample collocation points every Nchange iterations
Nint = 8000           # Number of interior collocation points (for PDE residual)
N0 = 500              # Number of initial-condition points (t=0)
Nb = 500              # Number of boundary points (x=-1 or x=1)
Nprint = 100          # Print loss every Nprint iterations

# --- Residual-based Adaptive Distribution (RAD) parameters ---
# RAD resamples collocation points so that regions with high PDE residual
# get more points — focusing computational effort where the network is worst.
k1 = 1   # Exponent: sampling weight = |residual|^k1
k2 = 1   # Additive constant: ensures even low-residual regions get some points
          # If k1=0, k2>0: uniform random sampling
          # If k1=1, k2=1: moderate focusing on high-residual regions
rad_args = (k1, k2)


# ========================
# SECTION 2: NEURAL NETWORK DEFINITION
# ========================

class CustomActivation(nn.Module):
    """
    A custom 'activation' that simply multiplies the input by `scaling`.
    When scaling=1, this is the identity function.
    Applied after the final linear layer so the output is: scaling * (W·z + b).
    This allows controlling the output magnitude without changing the loss.
    """
    def __init__(self, **kwargs):
        super().__init__()

    def forward(self, x):
        return x * scaling  # Element-wise multiplication by the scaling constant


def init_variance_scaling_fan_avg_uniform(linear, scale):
    """
    Custom weight initializer matching TensorFlow's VarianceScaling(mode='fan_avg').
    
    For a layer with fan_in input neurons and fan_out output neurons:
      n = (fan_in + fan_out) / 2
      limit = sqrt(3 * scale / n)
      weights ~ Uniform(-limit, limit)
      biases = 0
    
    This ensures the variance of activations stays roughly constant across layers,
    which is important for deep networks to avoid vanishing/exploding gradients.
    
    For the final layer, scale = 1/scaling² compensates for the CustomActivation
    that multiplies the output by `scaling`, keeping the overall output variance ~1.
    """
    fan_in, fan_out = linear.in_features, linear.out_features
    n = 0.5 * (fan_in + fan_out)        # Average of input and output dimensions
    limit = math.sqrt(3.0 * scale / n)  # Uniform distribution limit
    with torch.no_grad():
        linear.weight.uniform_(-limit, limit)  # Initialize weights uniformly in [-limit, limit]
        if linear.bias is not None:
            linear.bias.zero_()                # Initialize biases to zero


class Net(nn.Module):
    """
    The PINN neural network: a fully-connected MLP.
    
    Architecture:  (t, x) → Linear(2→20) → tanh → Linear(20→20) → tanh →
                            Linear(20→20) → tanh → Linear(20→20) → tanh →
                            Linear(20→1) → CustomActivation → u
    
    - 4 hidden layers with 20 neurons each
    - tanh activation (smooth, infinitely differentiable — important because
      we need to compute ∂u/∂t, ∂u/∂x, ∂²u/∂x² through automatic differentiation)
    - The final layer has custom initialization to control output scale
    
    Total trainable parameters: 2×20 + 20 + 20×20 + 20 + 20×20 + 20 + 20×20 + 20 + 20×1 + 1
                               = 40+20 + 400+20 + 400+20 + 400+20 + 20+1 = 1341
    """
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(layer_dims[0], layer_dims[1]),  # Linear: 2 → 20 (weights: 2×20, bias: 20)
            nn.Tanh(),                                 # tanh activation
            nn.Linear(layer_dims[1], layer_dims[2]),  # Linear: 20 → 20
            nn.Tanh(),                                 # tanh activation
            nn.Linear(layer_dims[2], layer_dims[3]),  # Linear: 20 → 20
            nn.Tanh(),                                 # tanh activation
            nn.Linear(layer_dims[3], layer_dims[4]),  # Linear: 20 → 20
            nn.Tanh(),                                 # tanh activation
            nn.Linear(layer_dims[4], layer_dims[5]),  # Linear: 20 → 1 (output layer)
            CustomActivation(),                        # Multiply output by `scaling`
        )
        # Custom-initialize the LAST linear layer (self.net[-2] because [-1] is CustomActivation)
        final_linear = self.net[-2]
        init_variance_scaling_fan_avg_uniform(final_linear, scale=1.0 / (scaling ** 2))

    def forward(self, x):
        """Forward pass: input x = (t, x) → predicted u(t,x)"""
        return self.net(x)


# Instantiate the network
N = Net()


# ========================
# SECTION 3: DATA SAMPLING FUNCTIONS
# ========================
# These functions generate the three types of training points:
# 1. Interior points (for PDE residual)
# 2. Initial-condition points (t=0)
# 3. Boundary points (x = -1 or x = 1)

def generate_inputs(Nint):
    """
    Generate Nint random interior collocation points in the domain [0,tfinal] × [x0,xf].
    These are where we enforce the PDE residual = 0.
    
    Returns: tensor of shape (Nint, 2), each row is (t, x)
    """
    t = tfinal * np.random.rand(Nint)         # Random t ∈ [0, 1]
    x = L * np.random.rand(Nint) + x0         # Random x ∈ [-1, 1]
    X = np.hstack((t[:, None], x[:, None]))    # Stack into (Nint, 2) array
    return torch.as_tensor(X)                  # Convert to PyTorch tensor


def initial_points(N0):
    """
    Generate N0 random initial-condition points at t = 0.
    These enforce u(0, x) = -sin(πx).
    
    Returns: tensor of shape (N0, 2), each row is (0, x)
    """
    t = np.zeros(N0)                           # All t = 0
    x = L * np.random.rand(N0) + x0           # Random x ∈ [-1, 1]
    X = np.hstack((t[:, None], x[:, None]))    # Stack into (N0, 2)
    return torch.as_tensor(X)


def boundary_points(Nb):
    """
    Generate Nb random boundary points at x = -1 or x = 1.
    These enforce u(t, -1) = 0 and u(t, 1) = 0.
    
    np.random.randint(0,2) gives 0 or 1; multiplied by L=2 and offset by x0=-1:
      0*2 + (-1) = -1  (left boundary)
      1*2 + (-1) =  1  (right boundary)
    
    Returns: tensor of shape (Nb, 2), each row is (t, ±1)
    """
    t = tfinal * np.random.rand(Nb)                     # Random t ∈ [0, 1]
    x = L * np.random.randint(0, 2, size=(Nb,)) + x0    # Randomly -1 or 1
    X = np.hstack((t[:, None], x[:, None]))
    return torch.as_tensor(X)


def generate_validation(Nt, Nx):
    """
    Generate a uniform grid of Nt × Nx points for validation / visualization.
    Returns the flattened tensor plus the mesh arrays for reshaping.
    """
    x = np.linspace(x0, xf, Nx)       # Nx evenly-spaced x values in [-1, 1]
    t = np.linspace(0, tfinal, Nt)     # Nt evenly-spaced t values in [0, 1]
    x, t = np.meshgrid(x, t)          # Create 2D grids, each shape (Nt, Nx)
    X = np.hstack((t.flatten()[:, None], x.flatten()[:, None]))  # Flatten to (Nt*Nx, 2)
    print('x', x.shape, 't', t.shape, 'X', X.shape)
    return torch.as_tensor(X), t, x


def adaptive_rad(N, Nint, rad_args, Nsampling=50000):
    """
    Residual-based Adaptive Distribution (RAD) — smart resampling of collocation points.
    
    Instead of uniform random sampling, RAD samples more points where the PDE residual
    is large (i.e., where the network is currently performing worst).
    
    Algorithm:
    1. Generate Nsampling candidate points uniformly
    2. Evaluate the PDE residual |f(t,x)| at each candidate
    3. Compute sampling probability:  p(x) ∝ |f(x)|^k1 / mean(|f|^k1) + k2
       - k1 controls how aggressively we focus on high-residual regions
       - k2 ensures all regions retain some probability (avoids zero-probability)
    4. Sample Nint points according to these probabilities (without replacement)
    
    This is one of the key techniques in the paper — adaptive resampling
    significantly improves convergence for problems with sharp features.
    """
    Xtest = generate_inputs(Nsampling)                              # 50,000 candidate points
    k1, k2 = rad_args                                               # Unpack RAD parameters
    Y = torch.abs(get_results(N, Xtest)[-1]).reshape(-1)            # |PDE residual| at each point
    w = (Y ** k1)                                                   # Weight = |residual|^k1
    err_eq = w / w.mean() + k2                                     # Normalize + add baseline k2
    p = (err_eq / err_eq.sum()).clamp_min(1e-12)                    # Convert to probability, clamp > 0
    X_ids = torch.multinomial(p, num_samples=Nint, replacement=False)  # Sample indices
    return Xtest[X_ids]                                             # Return selected points


# ========================
# SECTION 4: PDE CONDITIONS (IC and BC)
# ========================

def uinit(X):
    """
    Initial condition: u(0, x) = -sin(πx)
    
    This is the exact initial profile. The PINN must match this at t=0.
    At t=0: u = -sin(πx) is a smooth sine wave that evolves into a shock.
    """
    x = X[:, 1:2]                        # Extract x column, keep shape (N, 1)
    return -torch.sin(torch.pi * x)      # -sin(πx)


def uleft(X):
    """
    Left boundary condition: u(t, -1) = 0 for all t.
    Returns zeros tensor matching the number of left-boundary points.
    """
    Xleft = X[X[:, 1] == x0]             # Filter rows where x == -1
    t = Xleft[:, 0]                       # Get their t values
    return torch.zeros((t.shape[0], 1), dtype=torch.float64)  # Return column of zeros


def uright(X):
    """
    Right boundary condition: u(t, 1) = 0 for all t.
    Returns zeros tensor matching the number of right-boundary points.
    """
    Xright = X[X[:, 1] == xf]            # Filter rows where x == 1
    t = Xright[:, 0]
    return torch.zeros((t.shape[0], 1), dtype=torch.float64)


# ========================
# SECTION 5: FORWARD PASS AND PDE RESIDUAL COMPUTATION
# ========================

def output(N, X):
    """
    Simple forward pass through network N.
    Input: X of shape (batch, 2) = (t, x)
    Output: u of shape (batch, 1) = predicted solution
    """
    Nout = N(X)            # Forward pass: (batch, 2) → (batch, 1)
    u = Nout[:, 0:1]       # Extract first (only) output column, keep 2D shape
    return u


def get_results(N, X):
    """
    Compute the PDE residual f(t,x) = u_t + u·u_x − ν·u_xx at collocation points.
    
    This is the HEART of the PINN: we use automatic differentiation (autograd)
    to compute the derivatives of the neural network output with respect to its inputs.
    
    The Burgers' equation residual is:
        f = ∂u/∂t + u · ∂u/∂x − ν · ∂²u/∂x²
    
    If the network perfectly satisfies the PDE, then f = 0 everywhere.
    
    Steps:
    1. Forward pass: u = N(t, x)
    2. First derivatives via autograd: ∂u/∂t, ∂u/∂x
    3. Second derivative via autograd: ∂²u/∂x² = ∂(∂u/∂x)/∂x
    4. Assemble residual: f = u_t + u*u_x - ν*u_xx
    """
    X.requires_grad_(True)                    # Enable gradient tracking on input (t, x)
    u = output(N, X)                          # Step 1: u = N(t, x), shape (batch, 1)

    # Step 2: Compute ∂u/∂t and ∂u/∂x simultaneously
    # torch.autograd.grad computes du/dX where X = (t, x)
    # grad_outputs=ones because u is a vector (one entry per sample)
    # create_graph=True: we'll differentiate again for u_xx
    grads = torch.autograd.grad(
        u, X,                                  # d(u) / d(X)
        grad_outputs=torch.ones_like(u),       # "seed" gradient (since u is not scalar)
        create_graph=True,                     # Keep graph for second derivative
        retain_graph=True)[0]                  # [0] because autograd.grad returns a tuple

    u_t = grads[:, 0]                          # ∂u/∂t — first column of gradient
    u_x = grads[:, 1]                          # ∂u/∂x — second column of gradient

    # Step 3: Compute ∂²u/∂x² = ∂(u_x)/∂x
    # We differentiate u_x with respect to X, then take the x-component (column 1)
    u_xx = torch.autograd.grad(
        u_x, X,                                # d(u_x) / d(X)
        grad_outputs=torch.ones_like(u_x),     # Seed gradient
        create_graph=True,                     # Keep graph for backprop through loss
        retain_graph=True)[0][:, 1]            # Take x-component → ∂²u/∂x²

    # Step 4: Assemble the PDE residual
    # Burgers' equation: u_t + u * u_x - ν * u_xx = 0
    # If the network satisfies the PDE, fu ≈ 0
    fu = u_t + u[:, 0] * u_x - nu * u_xx      # u[:,0] to match shapes (flatten u from (N,1) to (N,))

    return u, fu  # Return prediction and residual


# ========================
# SECTION 6: LOSS FUNCTION
# ========================

loss_function = nn.MSELoss()  # Mean Squared Error: MSE(a, b) = mean((a - b)²)

# Loss weighting factors
lam0 = 5.0    # Weight for initial-condition loss (λ₀)
lamB = 5.0    # Weight for boundary-condition loss (λ_B)
# PDE residual loss has implicit weight = 1.0
# Higher weights on IC/BC force the network to satisfy boundary data more strictly


def loss(fu, u0, u0pinn, ul, ulpinn, ur, urpinn):
    """
    Total PINN loss = L_PDE + λ₀ · L_IC + λ_B · (L_BC_left + L_BC_right)
    
    Arguments:
        fu      : PDE residual at interior points, should be ≈ 0
        u0      : True initial condition u(0,x) = -sin(πx)
        u0pinn  : Network prediction at t=0: N(0,x)
        ul      : True left BC = 0
        ulpinn  : Network prediction at x=-1: N(t,-1)
        ur      : True right BC = 0
        urpinn  : Network prediction at x=1: N(t,1)
    
    Returns:
        Scalar loss value
    
    The math:
        L = (1/N_int) Σ [f(tᵢ,xᵢ)]²                           ← PDE residual
          + λ₀ · (1/N₀) Σ [N(0,xⱼ) - u₀(xⱼ)]²                ← Initial condition
          + λ_B · (1/N_b) Σ [N(tₖ,-1)]²                        ← Left BC
          + λ_B · (1/N_b) Σ [N(tₖ,+1)]²                        ← Right BC
    """
    Ntot = fu.shape[0]                                                 # Number of interior points
    zeros = torch.zeros((Ntot, 1), dtype=torch.get_default_dtype(),    # Target = 0 for PDE residual
                        device=fu.device)
    fu_col = fu.reshape(-1, 1)                                         # Reshape residual to column vector

    loss_value = (
        loss_function(fu_col, zeros)                                   # L_PDE = MSE(residual, 0)
        + lam0 * loss_function(u0, u0pinn)                             # λ₀ · L_IC
        + lamB * (loss_function(ul, ulpinn)                            # λ_B · L_BC_left
                  + loss_function(ur, urpinn))                         # λ_B · L_BC_right
    )
    return loss_value


# ========================
# SECTION 7: GRADIENT COMPUTATION AND ADAM TRAINING STEP
# ========================

def grads(N, X, X0, Xb):
    """
    Compute the loss and its gradient with respect to ALL network parameters.
    Used by the Adam optimizer (Phase 1).
    
    Steps:
    1. Zero existing gradients
    2. Forward pass → PDE residual + boundary predictions
    3. Compute total loss
    4. Backpropagate (loss.backward()) to fill .grad on each parameter
    5. Return parameter gradients and loss value
    """
    # Zero all parameter gradients from previous iteration
    for p in N.parameters():
        p.grad = None

    # Clone inputs and enable gradient tracking (needed for PDE derivative computation)
    X = X.clone().detach().requires_grad_(True)
    X0 = X0.clone().detach().requires_grad_(True)

    # Forward pass: compute PDE residual at interior points
    _, fu = get_results(N, X)

    # Evaluate IC: true vs predicted
    u0 = uinit(X0)           # True: u(0,x) = -sin(πx)
    u0pinn = output(N, X0)   # Predicted: N(0,x)

    # Evaluate BC: true vs predicted at left and right boundaries
    ul = uleft(Xb)                          # True left BC = 0
    ur = uright(Xb)                         # True right BC = 0
    mask_l = (Xb[:, 1] == x0)              # Boolean mask: which rows have x = -1
    mask_r = (Xb[:, 1] == xf)              # Boolean mask: which rows have x = 1
    ulpinn = output(N, Xb[mask_l])         # Network prediction at left boundary
    urpinn = output(N, Xb[mask_r])         # Network prediction at right boundary

    # Compute total PINN loss
    loss_value = loss(fu, u0, u0pinn, ul, ulpinn, ur, urpinn)

    # Backpropagation: compute ∂(loss)/∂(θ) for every parameter θ
    loss_value.backward()

    # Collect the computed gradients
    gradsN = [p.grad for p in N.parameters()]
    return gradsN, loss_value


def training(N, X, X0, Xb, optimizer):
    """
    One Adam training step:
    1. Compute gradients (which also fills param.grad via backward())
    2. optimizer.step() updates weights using Adam update rule:
         m_t = β₁·m_{t-1} + (1-β₁)·g_t          (first moment / mean)
         v_t = β₂·v_{t-1} + (1-β₂)·g_t²          (second moment / variance)
         θ_{t+1} = θ_t - lr · m̂_t / (√v̂_t + ε)   (parameter update)
    """
    _, loss_value = grads(N, X, X0, Xb)    # Compute loss + fill .grad
    optimizer.step()                         # Apply Adam update to all parameters
    return loss_value                        # Return loss for logging


# ========================
# SECTION 8: GENERATE INITIAL TRAINING DATA
# ========================

X  = generate_inputs(Nint)    # 8000 random interior points (t, x) for PDE residual
X0 = initial_points(N0)       # 500 random points at t=0 for initial condition
Xb = boundary_points(Nb)      # 500 random boundary points at x = ±1


# ========================
# SECTION 9: LOAD REFERENCE SOLUTION FOR VALIDATION
# ========================
# Load a high-fidelity reference solution from a MATLAB .mat file.
# This is used to compute the relative L² error and validate the PINN.

mat = scipy.io.loadmat("burgers_canonical.mat")  # Load .mat file
u_ref  = mat["usol"]          # Reference solution u(t,x), shape (Nt, Nx)
t_star = mat["t"].ravel()     # Time grid points
x_star = mat["x"].ravel()     # Space grid points
x_star, t_star = np.meshgrid(x_star, t_star)  # Create 2D mesh for evaluation

# Create a PyTorch tensor of all reference grid points for network evaluation
device = next(N.parameters()).device
X_star = torch.as_tensor(
    np.hstack((t_star.reshape(-1, 1), x_star.reshape(-1, 1))),
    dtype=torch.get_default_dtype(),
    device=device)

template = 'Epoch {}, loss: {}'  # Print template for Adam training


# ========================
# SECTION 10: PHASE 1 — ADAM TRAINING (FIRST-ORDER OPTIMIZER)
# ========================
# Adam is a first-order optimizer: it only uses the gradient (∂L/∂θ), no Hessian.
# It's fast per-iteration but converges slowly near the solution.
# We use it as a "warmup" to get the network into a roughly good region
# before switching to a more powerful second-order optimizer.

rel_adam = []   # Track relative L² error during Adam training

# Configure Adam optimizer
# lr=5e-3:       Learning rate (how big each step is)
# betas=(0.99, 0.999):  Exponential decay rates for the first/second moment estimates
# eps=1e-20:     Small constant for numerical stability in the denominator
optimizer = torch.optim.Adam(N.parameters(), lr=5e-3, betas=(0.99, 0.999), eps=1e-20)

# Learning rate scheduler: exponentially decay the LR
# After `step` steps: lr_effective = 5e-3 × 0.98^(step/1000)
# This means every 1000 steps, the LR decreases by 2%
adam_scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lr_lambda=lambda step: 0.98 ** (step / 1000.0))

adam_losses = []                   # Store loss at every Adam epoch
start_time = perf_counter()        # Start timer

adam_t0 = perf_counter()
for epoch in range(Nepochs_ADAM):  # 1000 Adam iterations
    # Every Nchange=500 epochs, resample collocation points using RAD
    # This moves points to where the residual is largest (adaptive refinement)
    if (epoch + 1) % Nchange == 0:
        X  = adaptive_rad(N, Nint, rad_args)   # Resample interior points adaptively
        X0 = initial_points(N0)                # Fresh random IC points
        Xb = boundary_points(Nb)               # Fresh random BC points

    # One training step: forward pass → loss → backward → Adam update
    loss_value = training(N, X, X0, Xb, optimizer)
    adam_losses.append(float(loss_value.detach().cpu().item()))

    # Every Nprint=100 epochs, print loss and compute validation error
    if (epoch + 1) % Nprint == 0:
        print(template.format(epoch + 1, adam_losses[-1]))

        # Compute relative L² error against the reference solution
        N.eval()                              # Set to eval mode (disables dropout, etc.)
        with torch.no_grad():                 # No need to track gradients for validation
            u_pred = output(N, X_star).detach().cpu().numpy().reshape(u_ref.shape)
        N.train()                             # Back to training mode

        # Relative L² error = ||u_ref - u_pred|| / ||u_pred||
        num = np.linalg.norm(u_ref - u_pred)        # Numerator: error norm
        den = np.linalg.norm(u_pred) + 1e-12        # Denominator: prediction norm (+ epsilon)
        rel = num / den                              # Relative error
        rel_adam.append((epoch + 1, float(rel)))     # Store for plotting


adam_time_sec = perf_counter() - adam_t0  # Total Adam training time


# ========================
# SECTION 11: NESTED_TENSOR UTILITY
# ========================
# This function reshapes a flat 1D weight vector into the nested list-of-arrays
# format that PyTorch/TensorFlow expect when setting network weights.
# It's used by the SciPy BFGS interface, which works with flat vectors.

def nested_tensor(grad, layer_dims, train_activations=False, bias=True):
    """
    Convert a flat 1D weight vector into a list of weight matrices and bias vectors,
    matching the layer dimensions defined by `layer_dims`.
    
    For layer_dims = (2, 20, 20, 20, 20, 1):
      - 5 weight matrices: (2×20), (20×20), (20×20), (20×20), (20×1)
      - 5 bias vectors:    (20,), (20,), (20,), (20,), (1,)
      - Total: 10 arrays, interleaved as [W1, b1, W2, b2, ..., W5, b5]
    
    This is needed because SciPy optimizers work with a single flat vector of all
    parameters, but PyTorch/TensorFlow need structured weight tensors.
    """
    if _has_torch and isinstance(grad, torch.Tensor):
        grad = grad.detach().cpu().numpy()
    grad = np.asarray(grad).ravel()              # Ensure it's a flat numpy array

    if not train_activations:
        if bias:
            # 2 arrays per layer transition: weight matrix + bias vector
            # For 6 layer_dims (5 transitions): 2*5 = 10 arrays
            temp = [None] * (2 * len(layer_dims) - 2)
        else:
            temp = [None] * (2 * len(layer_dims) - 3)

        index = 0  # Current position in the flat vector
        for i in range(len(temp)):
            if i % 2 == 0:
                # Even indices: weight matrices
                k = i // 2                                        # Layer index
                fan_in, fan_out = layer_dims[k], layer_dims[k + 1]
                expected_shape = (fan_in, fan_out)
                size = fan_in * fan_out                           # Number of weight elements

                print('layer_dims', layer_dims)
                print(f"Expected shape: {expected_shape}, Grad slice size: {size}")
                print(f"Current index: {index}, Next index: {index + size}")
                print(f"Size of grad slice: {grad[index:index+size].size}")

                temp[i] = grad[index:index + size].reshape(expected_shape)
                index += size
            else:
                # Odd indices: bias vectors
                k = i - (i // 2)                                  # Determine which layer's bias
                bsz = layer_dims[k]                               # Bias size = number of neurons
                temp[i] = grad[index:index + bsz]
                index += bsz
        return temp

    else:
        # Alternative layout when trainable activations are present (3 arrays per layer)
        temp = [None] * (3 * len(layer_dims) - 4)
        index = 0
        for i in range(len(temp)):
            if i % 3 == 0:
                # Weight matrix
                k = i // 3
                fan_in, fan_out = layer_dims[k], layer_dims[k + 1]
                size = fan_in * fan_out
                temp[i] = grad[index:index + size].reshape(fan_in, fan_out)
                index += size
            elif i % 3 == 1:
                # Bias vector
                k = int((i + 2) / 3)
                bsz = layer_dims[k]
                temp[i] = grad[index:index + bsz]
                index += bsz
            else:
                # Activation parameter (single scalar)
                temp[i] = grad[index]
                index += 1
        return temp


# ========================
# SECTION 12: SCIPY-COMPATIBLE LOSS+GRADIENT WRAPPER
# ========================
# SciPy's minimize() expects a function f(x) → (loss, gradient) where x is a
# flat numpy array of all parameters. These wrappers bridge PyTorch ↔ SciPy.

power = 1.0  # Loss power transform: L_effective = L^(1/power). When power=1, no transform.


def loss_and_gradient_torch(N, X, X0, Xb, power=power):
    """
    Compute loss and gradients using PyTorch autograd, for the BFGS phase.
    
    Unlike the grads() function used for Adam (which uses .backward()),
    this uses torch.autograd.grad() to get gradients directly as a list,
    without populating .grad attributes.
    
    The `power` parameter allows optimizing L^(1/power) instead of L.
    When power=1: minimize L directly.
    When power=2: minimize √L (can help with conditioning).
    """
    X = X.clone().detach().requires_grad_(True)  # Fresh copy with gradient tracking
    _, fu = get_results(N, X)                     # Forward pass + PDE residual

    # Evaluate IC and BC predictions
    u0 = uinit(X0)
    u0pinn = output(N, X0)
    ul = uleft(Xb)
    mask_l = (Xb[:, 1] == x0)
    ulpinn = output(N, Xb[mask_l])
    ur = uright(Xb)
    mask_r = (Xb[:, 1] == xf)
    urpinn = output(N, Xb[mask_r])

    # Compute total loss
    loss_value = loss(fu, u0, u0pinn, ul, ulpinn, ur, urpinn)

    # Optionally transform: L^(1/power)
    loss_root = loss_value if power == 1.0 else loss_value ** (1.0 / power)

    # Compute gradients of loss w.r.t. ALL network parameters
    params = list(N.parameters())
    gradsN = torch.autograd.grad(
        loss_root, params,
        create_graph=False,       # No higher-order derivatives needed here
        retain_graph=False,       # Free memory after this
        allow_unused=False)       # All params must contribute to loss
    return loss_root, gradsN


def loss_and_gradient(weights, N, X, X0, Xb, layer_dims=None):
    """
    SciPy-compatible interface: flat numpy vector → (loss_scalar, gradient_array).
    
    This is the function passed to scipy.optimize.minimize().
    
    Steps:
    1. Convert the flat numpy weight vector to a PyTorch tensor
    2. Load those weights into the network (vector_to_parameters)
    3. Compute loss and gradients using PyTorch
    4. Flatten gradients back to a numpy array for SciPy
    
    The jac=True flag in minimize() tells SciPy that this function returns
    both the loss AND the gradient in one call (more efficient).
    """
    first_param = next(N.parameters())
    device = first_param.device
    dtype = first_param.dtype

    # Convert flat numpy weights to PyTorch tensor
    w_tensor = torch.as_tensor(weights, dtype=dtype, device=device)

    # Load weights into network (overwrite current parameters)
    with torch.no_grad():
        vector_to_parameters(w_tensor, N.parameters())

    # Compute loss and gradient
    loss_val, grads_list = loss_and_gradient_torch(N, X, X0, Xb, power=power)

    # Flatten all parameter gradients into one numpy array
    grads_flat = torch.cat([g.reshape(-1) for g in grads_list])

    # Return (scalar loss, flat gradient array) — both as numpy
    return float(loss_val.detach().cpu().item()), grads_flat.detach().cpu().numpy()


# ========================
# SECTION 13: PHASE 2 SETUP — SELF-SCALED BFGS
# ========================
# After Adam warmup, we switch to a quasi-Newton optimizer (SSBroyden2).
#
# WHY SWITCH?
# - Adam uses only first-order info (gradients). Near the solution, it's slow.
# - BFGS builds an approximation H_k ≈ (∂²L/∂θ²)⁻¹ (inverse Hessian).
# - It uses this to take much larger, better-directed steps:  θ_{k+1} = θ_k - H_k · ∇L
# - Self-scaled variants (SSBFGS, SSBroyden) improve the Hessian update by
#   multiplying H_k by a scaling factor τ_k before each rank-2 update.
# - This keeps the condition number of H_k controlled and improves convergence.

Nbfgs = 10000                           # Total BFGS iterations (budget)
Nbatches = int(round(Nbfgs / Nchange))  # Number of batches (each Nchange=500 iters)
Nprint = 100                             # Print every 100 iterations

# Tracking arrays for checkpoints
n_ckpts = Nbfgs // Nprint                                  # 100 checkpoints total
warmup_steps = len(adam_losses)                             # Number of Adam steps completed
epochs_bfgs = warmup_steps + np.arange(1, n_ckpts + 1, dtype=float) * Nprint  # Epoch numbers
lossbfgs = np.zeros(n_ckpts, dtype=float)                  # Training loss at each checkpoint
validation_list = np.zeros(n_ckpts, dtype=float)           # Validation loss
error_list = np.zeros(n_ckpts, dtype=float)                # Relative L² error
time_elapsed = np.array([])                                 # Time tracking
initial_time = perf_counter()

# Create a validation grid (300×300 = 90,000 points)
Nt = 300
Nx = 300
Xtest, t, x = generate_validation(Nt, Nx)
X0test = Xtest[:Nx]                                         # First Nx rows are at t=0
mask_b = (Xtest[:, 1] == x0) | (Xtest[:, 1] == xf)        # Boundary mask
Xbtest = Xtest[mask_b]                                      # Boundary points

# Flatten all network parameters into a single numpy vector (starting point for BFGS)
initial_weights = parameters_to_vector([p.detach() for p in N.parameters()]).cpu().numpy()

# BFGS iteration counter (persists across multiple minimize() calls)
cont = 0


def callback(*, intermediate_result):
    """
    Called by SciPy after each BFGS iteration.
    Every Nprint=100 iterations, this:
    1. Records the training loss
    2. Computes validation loss (on a separate test grid)
    3. Computes the relative L² error vs the reference solution
    4. Prints a progress report
    
    The `intermediate_result` object has a `.fun` attribute = current loss value.
    """
    global N, cont, lossbfgs, Nprint, u_ref, \
           x_star, X_star, Xtest, validation_list, X0test, Xbtest, error_list, power

    device = next(N.parameters()).device
    dtype = next(N.parameters()).dtype

    cont += 1                                     # Increment global iteration counter
    if (cont % Nprint) != 0:                      # Only log every Nprint iterations
        return

    idx = cont // Nprint - 1                      # Index into tracking arrays
    if idx < 0 or idx >= len(lossbfgs):           # Bounds check
        return

    # Record training loss (undo the power transform if power ≠ 1)
    loss_value = float((intermediate_result.fun) ** power)
    lossbfgs[idx] = loss_value

    # Compute validation: PDE residual on the test grid
    _, futest = get_results(N, Xtest.to(device))
    zeros = torch.zeros((futest.shape[0], 1), dtype=dtype, device=device)

    with torch.no_grad():
        # Predict u on the reference grid
        u_pred = output(N, X_star).reshape(x_star.shape)
        u_ref_t = torch.as_tensor(u_ref, dtype=dtype, device=device)

        # Validation loss = MSE(u_pred, u_ref) + MSE(residual, 0)
        v1 = loss_function(u_pred, u_ref_t)               # Data fit error
        v2 = loss_function(futest.reshape(-1, 1), zeros)   # Residual error
        validation_list[idx] = (v1 + v2).item()

    # Relative L² error = ||u_ref - u_pred|| / ||u_pred||
    u_np = u_pred.detach().cpu().numpy()
    num = np.linalg.norm(u_ref - u_np)
    den = np.linalg.norm(u_np) + 1e-12
    error_list[idx] = num / den

    print(f"Iteration {cont} (ckpt {idx+1}/{len(lossbfgs)}): "
          f"Train {loss_value:.3e}, relL2 {error_list[idx]:.3e}")


# ========================
# SECTION 14: BFGS OPTIMIZER CONFIGURATION
# ========================

method = "BFGS"                    # Tell SciPy to use the BFGS algorithm
method_bfgs = "SSBroyden2"        # Tell the *patched* SciPy to use Self-Scaled Broyden variant 2
                                   # This is one of the paper's proposed improvements:
                                   # a one-parameter Broyden family with a clamped θ_k choice
initial_scale = False              # Don't apply Nocedal-Wright initial scaling to H₀
warmup_tag = "warmup_adam"         # Label for logging purposes

os.makedirs("results2", exist_ok=True)  # Ensure output directory exists


def _scipy_cb_factory():
    """
    Factory for an alternative callback style (old-style SciPy, not used in the main loop).
    Kept for compatibility.
    """
    class _Res:
        __slots__ = ("fun",)
        def __init__(self, fun): self.fun = fun
    def _cb(xk):
        f, _ = loss_and_gradient(xk, N, X, X0, Xb, layer_dims)
        callback(intermediate_result=_Res(f))
    return _cb


# Initialize the inverse Hessian approximation as the identity matrix.
# H₀ = I means the first BFGS step is just gradient descent: Δθ = -I · ∇L = -∇L
# As iterations proceed, H_k accumulates curvature info and steps become Newton-like.
H0 = np.eye(initial_weights.size, dtype=np.float64)

initial_time_bfgs = perf_counter()

bfgs_loss_ckpt = []  # Additional loss tracking lists
bfgs_rel = []

bfgs_t0 = perf_counter()  # BFGS phase start time


# ========================
# SECTION 15: PHASE 2 — SELF-SCALED BROYDEN TRAINING LOOP
# ========================
# This is the main second-order optimization loop.
#
# Key idea: we call scipy.optimize.minimize() in a LOOP, not just once.
# Each call runs at most Nchange=500 BFGS iterations, then we:
#   1. Extract the updated weights and inverse Hessian approximation
#   2. Resample collocation points (RAD) — this changes the loss landscape!
#   3. Continue optimizing with the new points and warm-started Hessian
#
# This "batched BFGS" approach is necessary because:
# - The collocation points define the loss function
# - Periodically resampling them (especially with RAD) prevents overfitting
#   to a fixed set of points and improves generalization
# - The Hessian approximation H₀ is carried forward between calls (warm start),
#   so curvature info isn't lost

while cont < Nbfgs:  # Loop until we've done 10,000 total BFGS iterations
    print(cont)       # Print current iteration count

    # Call SciPy's minimize with the patched BFGS
    result = minimize(
        loss_and_gradient,            # Function: flat_weights → (loss, gradient)
        initial_weights,              # Starting point: flat numpy vector of all parameters
        args=(N, X, X0, Xb, layer_dims),  # Extra arguments passed to loss_and_gradient
        method=method,                # "BFGS" — but our patched version with method_bfgs
        jac=True,                     # Tell SciPy that loss_and_gradient returns (loss, grad)
        options={
            'maxiter': Nchange,       # Run at most 500 iterations before returning
            'gtol': 0,                # Gradient tolerance = 0 (never stop for small gradient)
            'hess_inv0': H0,          # Warm-start: use previous inverse Hessian approximation
            'method_bfgs': method_bfgs,    # "SSBroyden2" — the self-scaled Broyden update
            'initial_scale': initial_scale  # False — don't do initial H₀ scaling
        },
        tol=0,                        # Function tolerance = 0 (never stop for small f change)
        callback=callback             # Called after each iteration for logging
    )

    # Extract final weights from this batch (starting point for next call)
    initial_weights = result.x

    # Extract the inverse Hessian approximation (warm start for next batch)
    H0 = result.hess_inv
    # Symmetrize: H should be symmetric but floating-point can introduce tiny asymmetries
    H0 = 0.5 * (H0 + H0.T)

    # Verify H₀ is still positive-definite (required for BFGS to work correctly)
    # If it's not (numerical issues), reset to identity matrix
    try:
        cholesky(H0)                  # Cholesky decomposition succeeds iff H₀ > 0
    except LinAlgError:
        # H₀ lost positive-definiteness — reset to identity (lose curvature info)
        H0 = np.eye(len(initial_weights), dtype=np.float64)

    # Resample all training points using RAD (adaptive refinement)
    X  = adaptive_rad(N, Nint, rad_args)   # New interior points focused on high residual
    X0 = initial_points(N0)                # Fresh IC points
    Xb = boundary_points(Nb)               # Fresh BC points

    # Evaluate current loss on the new points (for monitoring)
    _, fu   = get_results(N, X)
    u0      = uinit(X0)
    u0pinn  = output(N, X0)
    ul      = uleft(Xb)
    ulpinn  = output(N, Xb[Xb[:, 1] == x0])
    ur      = uright(Xb)
    urpinn  = output(N, Xb[Xb[:, 1] == xf])
    loss_value = loss(fu, u0, u0pinn, ul, ulpinn, ur, urpinn)

    initial_scale = False  # Keep initial_scale off for subsequent batches

# Total BFGS training time
bfgs_time_sec = perf_counter() - bfgs_t0


Epoch 100, loss: 0.9645359335530801
Epoch 200, loss: 0.49671976261755996
Epoch 300, loss: 0.3869582024360727
Epoch 400, loss: 0.321966538914848
Epoch 500, loss: 0.40809901477749017
Epoch 600, loss: 0.369119024459356
Epoch 700, loss: 0.33958188810950274
Epoch 800, loss: 0.311052270314498
Epoch 900, loss: 0.2666776018277991
Epoch 1000, loss: 0.330460989942149
x (300, 300) t (300, 300) X (90000, 2)
0
Iteration 100 (ckpt 1/100): Train 1.108e-02, relL2 1.048e-01
Iteration 200 (ckpt 2/100): Train 6.538e-04, relL2 3.258e-02
Iteration 300 (ckpt 3/100): Train 8.764e-05, relL2 4.143e-03
Iteration 400 (ckpt 4/100): Train 2.861e-05, relL2 3.255e-03
Iteration 500 (ckpt 5/100): Train 1.260e-05, relL2 1.641e-03
500
Iteration 600 (ckpt 6/100): Train 1.264e-05, relL2 9.923e-04
Iteration 700 (ckpt 7/100): Train 5.051e-06, relL2 3.307e-04
Iteration 800 (ckpt 8/100): Train 2.457e-06, relL2 2.271e-04
Iteration 900 (ckpt 9/100): Train 1.286e-06, relL2 1.274e-04
Iteration 1000 (ckpt 10/100): Train 7.455e-07,